In [3]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [ ]:
from scraper.cricbuzz import scrape_cricbuzz_news
from scraper.espn import scrape_espn_news
from rag.ingestion import ingest_document
from config.constants import INTENT_SMART_ALERT
import logging
from agents.supervisor import get_agent
from apscheduler.schedulers.asyncio import AsyncIOScheduler
from tools.mongodb_tools import fetch_live_matches

ModuleNotFoundError: No module named 'tools'

In [5]:
logger=logging.getLogger(__name__)
scheduler=AsyncIOScheduler()

In [10]:
async def scrape_and_ingest():
    scrapers=[scrape_cricbuzz_news,scrape_espn_news]
    for scraper in scrapers:
        try:
            articles=await scraper()
            for a in articles:
                if a.get("content"):
                    await ingest_document(
                        source=scraper.__name__,
                        url=a.get("url"),
                        title=a.get("title"),
                        content=a.get("content"),
                        category=a.get("category"),
                    )
            logger.info(f"Scraped {len(articles)}")
        except Exception as e:
            logger.error(f"Scraper error {e}")

In [11]:
await scrape_and_ingest()

Scraper error Database not connected


In [12]:
async def run_smart_alerts():
    agent=get_agent()
    live=await fetch_live_matches()
    for match in live:
        try:
            await agent.ainvoke({
                "query":"Generate smart alert for live match",
                "intent":INTENT_SMART_ALERT,
                "intent_confidence":1.0,
                "intent_slots":{},
                "context":{
                    "matchId":str(match["_id"]),
                    "pageType":"match"
                },
                "match_data":match,
                "live_score":match.get("liveScore",{}),
                "messages":[]
            })
        except Exception as e:
            logger.error(f"Alert error {e}")


In [13]:
from config.settings import settings
def start_scheduler():
    scheduler.add_job(scrape_and_ingest,"interval",
    minutes=settings.SCRAPE_INTERVAL,id="scrape")
    scheduler.add_job(run_smart_alerts,"interval",minutes=5,id="alerts")
    scheduler.start()
    logger.info("scheduler started")
    

In [ ]:
def stop_scheduler():
    scheduler.shutdown()